# Phase 3 — Architecture inspection + LoRA smoke test ONLY

**Scope, per the frozen Phase-3 protocol Sections C and G:**
1. Inspect the actual loaded Qwen2.5-Omni-3B module hierarchy; determine the exact LoRA
   `target_modules` specification empirically (not assumed).
2. Apply LoRA with that spec; prove no audio/vision/talker parameter is trainable; report
   the exact trainable-parameter count.
3. Run the minimal fp16 LoRA smoke test: multi-T4 placement, forward/loss, backward,
   LoRA-only gradients, optimizer step, checkpoint save/fresh-load/resume, post-training
   generation.

**This notebook does NOT run the full Session 1-3 training job and does NOT read, write,
or reference Session 5 anywhere.** It reuses the Sessions-1-4 dataset already uploaded and
validated for Experiment 4 (`kaggle_upload_s14calib`) — no new data build.

If any smoke-test gate fails, this notebook follows the frozen escalation order (hand-specified
device map → single-GPU) and reports explicitly which rung was needed. It does not silently
change architecture, quantization, target modules, or the training protocol.


In [1]:
# ============================ CONFIG ============================
MODEL_ID   = "Qwen/Qwen2.5-Omni-3B"
SEED       = 11
N_SMOKE    = 4            # "a handful of examples" per the frozen protocol -- minimal, not the real training set
MAX_NEW_TOKENS = 12        # identical generation setting to Experiments 2-4, for the post-training gen check
OUTPUT_DIR = "/kaggle/working/phase3_smoketest"
EVAL_LIB_SHA256_EXPECTED = "24fac1f49d6f049c1485b4769ce523a292ae692fad4657a9fab7be6af6034298"
print("MODEL_ID:", MODEL_ID, "| N_SMOKE:", N_SMOKE)
print("SCOPE: architecture inspection + smoke test ONLY. No full training. No Session 5.")


MODEL_ID: Qwen/Qwen2.5-Omni-3B | N_SMOKE: 4
SCOPE: architecture inspection + smoke test ONLY. No full training. No Session 5.


In [2]:
import subprocess, sys
import numpy as _np0, scipy as _sp0
_pin_path = "/kaggle/working/_pin_numpy_scipy.txt"
with open(_pin_path, "w") as _f:
    _f.write("numpy==" + _np0.__version__ + "\n")
    _f.write("scipy==" + _sp0.__version__ + "\n")
print("pinning numpy/scipy for every install below to:", _np0.__version__, _sp0.__version__)
def pip(*a): subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-c", _pin_path, *a], check=True)
pip("-U", "transformers>=4.52.3", "accelerate>=0.34", "qwen-omni-utils", "librosa", "soundfile", "peft>=0.11.1")
# NOTE: deliberately NOT reinstalling numpy/scipy. An earlier version of this cell force-
# reinstalled them to "whatever pip resolves as latest today" (a fix carried over from a
# different Kaggle image snapshot, for Experiment 4), which produced an internally-inconsistent
# numpy install here (ImportError: cannot import name '_center' from numpy._core.umath),
# reachable from multiple unrelated import paths inside transformers/scipy -- not something we
# could dodge one trigger at a time. Kaggle's own pre-installed numpy/scipy pairing is presumably
# self-consistent (shipped and tested as a coherent set); reverting to it and relying on the
# narrow, already-present _blas_supports_fpe shim below (a no-op if not needed) is the safer
# choice over re-introducing an unpinned reinstall.
import numpy as _np_check, scipy as _sp_check
print("numpy  :", _np_check.__version__, "->", _np_check.__file__)
print("scipy  :", _sp_check.__version__, "->", _sp_check.__file__)
# Kaggle's base image ships a stale torchao (0.10.0) that peft's LoRA module-dispatch chain
# unconditionally probes when wrapping ANY target module (even though we never use torchao/
# quantization) -- peft's is_torchao_available() raises rather than skipping on a too-old
# install. Purely an environment fix; does not enable or touch quantization.
pip("-U", "torchao>=0.16.0")
print("pip done")


pinning numpy/scipy for every install below to: 2.0.2 1.16.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 40.3 MB/s eta 0:00:00
numpy  : 2.0.2 -> /usr/local/lib/python3.12/dist-packages/numpy/__init__.py
scipy  : 1.16.3 -> /usr/local/lib/python3.12/dist-packages/scipy/__init__.py
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 10.1 MB/s eta 0:00:00
pip done


In [3]:
import os, json, time, random, hashlib, base64
import numpy as np
# numpy/scipy private-API compat shim (same fix required in Experiment 4's Kaggle image)
import numpy._core._multiarray_umath as _mu
if not hasattr(_mu, "_blas_supports_fpe"):
    _mu._blas_supports_fpe = lambda *a, **k: False
import torch

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

import transformers, peft
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("peft        :", peft.__version__)
print("CUDA        :", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  cuda:{i} = {torch.cuda.get_device_name(i)}, "
          f"{torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB")
DTYPE = torch.float16  # T4s -> fp16, matching every prior notebook in this project
os.makedirs(OUTPUT_DIR, exist_ok=True)


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


torch       : 2.10.0+cu128
transformers: 5.17.0
peft        : 0.21.0
CUDA        : True | GPUs: 2
  cuda:0 = Tesla T4, 15.6 GB
  cuda:1 = Tesla T4, 15.6 GB


In [4]:
# ---- locate the Sessions-1-4 calibration dataset (Experiment 4's dataset, reused as-is) ----
CANDIDATE_ROOTS = []
for base in ["/kaggle/input"]:
    if os.path.isdir(base):
        for d in sorted(os.listdir(base)):
            CANDIDATE_ROOTS.append(os.path.join(base, d))

REQUIRED_FILES = ["s14_calib_tiny_nohist.json", "s14_calib_tiny_hist3.json"]

def _find_input_dir():
    for root in CANDIDATE_ROOTS:
        for dirpath, _, files in os.walk(root):
            if all(f in files for f in REQUIRED_FILES) and os.path.isdir(os.path.join(dirpath, "audio")):
                return dirpath
    raise FileNotFoundError(
        f"Could not find {REQUIRED_FILES} + audio/ under /kaggle/input. Add the SAME "
        f"S1-4 calibration dataset used for Experiment 4. Seen roots: {CANDIDATE_ROOTS}")

INPUT_DIR = _find_input_dir()
AUDIO_DIR = os.path.join(INPUT_DIR, "audio")
print("INPUT_DIR:", INPUT_DIR)
assert "test.json" not in os.listdir(INPUT_DIR), "This looks like the Session-5 dataset -- STOP."

def load_records(name):
    with open(os.path.join(INPUT_DIR, name), encoding="utf-8") as f:
        return json.load(f)

nohist_tiny = load_records("s14_calib_tiny_nohist.json")[:N_SMOKE]
hist3_tiny  = load_records("s14_calib_tiny_hist3.json")[:N_SMOKE]
assert [r["id"] for r in nohist_tiny] == [r["id"] for r in hist3_tiny]
sessions_seen = {r["session"] for r in nohist_tiny}
assert sessions_seen <= {1, 2, 3, 4}, f"non-S1-4 session found: {sessions_seen}"
print(f"smoke batch: {len(nohist_tiny)} examples, sessions={sorted(sessions_seen)} (S1-4 only, confirmed)")
for r in nohist_tiny:
    print(" ", r["id"], "gold=", r["output"])


INPUT_DIR: /kaggle/input/datasets/pranavjaiganesh/kaggle-experiment4-s14-calibration/kaggle_upload_s14calib
smoke batch: 4 examples, sessions=[1] (S1-4 only, confirmed)
  Ses01F_impro03_F012 gold= happy
  Ses01F_impro03_F013 gold= happy
  Ses01F_impro03_M014 gold= happy
  Ses01F_impro03_F016 gold= happy


In [5]:
# ---- HF auth (optional; Qwen is ungated) ----
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets.")
except Exception as e:
    print("No HF_TOKEN secret (fine for Qwen):", e)


HF_TOKEN loaded from Kaggle Secrets.


In [6]:
# ================= iemocap_eval_lib.py, embedded verbatim as base64 =================
# UNCHANGED from Experiments 2-4 -- reused here only for build_prompt() (prompt construction),
# not for scoring (no evaluation happens in this smoke test).
EVAL_LIB_B64 = "..."
_lib_bytes = base64.b64decode(EVAL_LIB_B64)
_lib_sha256 = hashlib.sha256(_lib_bytes).hexdigest()
assert _lib_sha256 == EVAL_LIB_SHA256_EXPECTED, "iemocap_eval_lib.py mismatch -- STOP"
with open("/kaggle/working/iemocap_eval_lib.py", "wb") as f:
    f.write(_lib_bytes)
import importlib, sys
sys.path.insert(0, "/kaggle/working")
import iemocap_eval_lib as EVAL
importlib.reload(EVAL)
print("eval lib ready (sha256 matches Experiments 2-4). labels:", EVAL.IEMOCAP_LABELS)


eval lib ready (sha256 matches Experiments 2-4). labels: ['happy', 'sad', 'neutral', 'angry', 'excited', 'frustrated']


In [7]:
# ================= SECTION C: ARCHITECTURE INSPECTION =================
from transformers import Qwen2_5OmniProcessor
import transformers as _tf

OmniCls = getattr(_tf, "Qwen2_5OmniForConditionalGeneration", getattr(_tf, "Qwen2_5OmniModel", None))
if OmniCls is None:
    raise ImportError("This transformers build has no Qwen2.5-Omni class.")

proc = Qwen2_5OmniProcessor.from_pretrained(MODEL_ID)
base_model = OmniCls.from_pretrained(MODEL_ID, torch_dtype=DTYPE, device_map="auto")
base_model.eval()
print("base model loaded:", MODEL_ID)

print("\n--- device map (multi-T4 placement check happens here, at load time) ---")
hf_dm = getattr(base_model, "hf_device_map", None)
print("hf_device_map entries:", len(hf_dm) if hf_dm else 0)
devices_used = sorted(set(str(v) for v in hf_dm.values())) if hf_dm else []
print("distinct devices used:", devices_used)

print("\n--- top-level named children of the loaded model ---")
top_level = [name for name, _ in base_model.named_children()]
print(top_level)

print("\n--- full named_modules() count and a grep for *_proj, grouped by top-level prefix ---")
all_modules = list(base_model.named_modules())
print("total named modules:", len(all_modules))

import re, collections
proj_matches = [name for name, _ in all_modules if re.search(r"(^|\.)(q_proj|k_proj|v_proj|o_proj)$", name)]
by_prefix = collections.defaultdict(list)
for name in proj_matches:
    top = name.split(".")[0]
    by_prefix[top].append(name)

print(f"\ntotal q/k/v/o_proj matches: {len(proj_matches)}")
for top, names in sorted(by_prefix.items()):
    print(f"  prefix '{top}': {len(names)} matches, e.g. {names[:2]}")

# thinker decoder-layer count / naming pattern
thinker_attn = sorted(n for n in proj_matches if n.startswith("thinker."))
layer_ids = sorted(set(re.findall(r"\.layers\.(\d+)\.", " ".join(thinker_attn))), key=int)
print(f"\nthinker attention module examples: {thinker_attn[:8]}")
print(f"thinker decoder layer indices found: {layer_ids[:5]} ... {layer_ids[-3:] if layer_ids else []} "
      f"(n_layers={len(layer_ids)})")


chat_template.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/667 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

[transformers] Model config: tts_text_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151860. This may result in unexpected behavior.
[transformers] Model config: tts_text_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151861. This may result in unexpected behavior.
[transformers] Model config: tts_text_pad_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151859. This may result in unexpected behavior.
[transformers] Model config: vision_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151652. This may result in unexpected behavior.
[transformers] Model config: vision_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151653. This may result in unexpected behavior.
[transformers] Model config: audio_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/2543 [00:00<?, ?it/s]

[transformers] Qwen2_5OmniForConditionalGeneration LOAD REPORT from: Qwen/Qwen2.5-Omni-3B
Key                                                | Status     |  | 
---------------------------------------------------+------------+--+-
token2wav.code2wav_dit_model.rotary_embed.inv_freq | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

spk_dict.pt:   0%|          | 0.00/260k [00:00<?, ?B/s]

base model loaded: Qwen/Qwen2.5-Omni-3B

--- device map (multi-T4 placement check happens here, at load time) ---
hf_device_map entries: 44
distinct devices used: ['0', '1']

--- top-level named children of the loaded model ---
['thinker', 'talker', 'token2wav']

--- full named_modules() count and a grep for *_proj, grouped by top-level prefix ---
total named modules: 2720

total q/k/v/o_proj matches: 336
  prefix 'talker': 96 matches, e.g. ['talker.model.layers.0.self_attn.q_proj', 'talker.model.layers.0.self_attn.k_proj']
  prefix 'thinker': 240 matches, e.g. ['thinker.audio_tower.layers.0.self_attn.k_proj', 'thinker.audio_tower.layers.0.self_attn.v_proj']

thinker attention module examples: ['thinker.audio_tower.layers.0.self_attn.k_proj', 'thinker.audio_tower.layers.0.self_attn.q_proj', 'thinker.audio_tower.layers.0.self_attn.v_proj', 'thinker.audio_tower.layers.1.self_attn.k_proj', 'thinker.audio_tower.layers.1.self_attn.q_proj', 'thinker.audio_tower.layers.1.self_attn.v_proj', 't

In [8]:
# ---- decide the EXACT target_modules spec (thinker.* is NOT a safe prefix by itself --
# thinker contains BOTH the audio encoder (thinker.audio_tower.*) and the LLM decoder
# (thinker.model.*) as sub-modules, so a bare startswith("thinker.") filter would wrongly
# include the audio tower. Checked here at full nested granularity, not just the first
# dot-segment, and asserted directly against the raw match list.) ----
print("="*78)
print("TARGET-MODULE DECISION")
print("="*78)

nested_prefixes = sorted(set(".".join(n.split(".")[:2]) for n in proj_matches))
print("all 2-segment prefixes containing q/k/v/o_proj matches:", nested_prefixes)

THINKER_LLM_PREFIX = "thinker.model."   # the decoder-only LLM stack, confirmed by inspection above
TARGET_MODULES = sorted(n for n in proj_matches if n.startswith(THINKER_LLM_PREFIX))

excluded_under_thinker = sorted(n for n in proj_matches
                                if n.startswith("thinker.") and not n.startswith(THINKER_LLM_PREFIX))
print(f"\nExcluded (nested under 'thinker.' but NOT the LLM decoder): {len(excluded_under_thinker)} matches")
print("  e.g.:", excluded_under_thinker[:3])

assert TARGET_MODULES, ("No thinker.model attention projections found -- STOP and report the "
                        "exact prefixes printed above; the architecture assumption needs revision.")
bad = [n for n in TARGET_MODULES if any(seg in n for seg in ["audio_tower", "visual", "talker", "token2wav"])]
assert not bad, f"target list still includes non-LLM pathway modules: {bad}"

layer_nums = sorted({int(re.search(r"\.layers\.(\d+)\.", n).group(1)) for n in TARGET_MODULES})
print(f"\nFinal TARGET_MODULES: {len(TARGET_MODULES)} paths, {len(layer_nums)} decoder layers "
      f"(indices {layer_nums[0]}..{layer_nums[-1]}), {len(TARGET_MODULES)//len(layer_nums)} proj-types/layer")
print("First 4:", TARGET_MODULES[:4])
print("Last 4 :", TARGET_MODULES[-4:])
print("\nPASS: TARGET_MODULES == thinker.model.* only; audio_tower/visual/talker/token2wav excluded.")


TARGET-MODULE DECISION
all 2-segment prefixes containing q/k/v/o_proj matches: ['talker.model', 'thinker.audio_tower', 'thinker.model']

Excluded (nested under 'thinker.' but NOT the LLM decoder): 96 matches
  e.g.: ['thinker.audio_tower.layers.0.self_attn.k_proj', 'thinker.audio_tower.layers.0.self_attn.q_proj', 'thinker.audio_tower.layers.0.self_attn.v_proj']

Final TARGET_MODULES: 144 paths, 36 decoder layers (indices 0..35), 4 proj-types/layer
First 4: ['thinker.model.layers.0.self_attn.k_proj', 'thinker.model.layers.0.self_attn.o_proj', 'thinker.model.layers.0.self_attn.q_proj', 'thinker.model.layers.0.self_attn.v_proj']
Last 4 : ['thinker.model.layers.9.self_attn.k_proj', 'thinker.model.layers.9.self_attn.o_proj', 'thinker.model.layers.9.self_attn.q_proj', 'thinker.model.layers.9.self_attn.v_proj']

PASS: TARGET_MODULES == thinker.model.* only; audio_tower/visual/talker/token2wav excluded.


In [9]:
# ================= apply LoRA with the empirically-derived target spec =================
from peft import LoraConfig, get_peft_model

# Frozen-protocol defaults (Section F), rank/alpha/dropout only -- target_modules is the
# empirically-derived list from the cell above, NOT a guessed short name.
lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=16, lora_alpha=16, lora_dropout=0.05,
    target_modules=TARGET_MODULES,
    bias="none",
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
frozen_trainable_violation = [n for n, p in trainable
                              if any(seg in n for seg in ["audio_tower", "visual", "talker", "token2wav"])]
n_trainable = sum(p.numel() for _, p in trainable)
n_total = sum(p.numel() for p in model.parameters())

print(f"\nEXACT trainable parameter count: {n_trainable:,} / {n_total:,} total "
      f"({100*n_trainable/n_total:.4f}%)")
print(f"Number of distinct trainable parameter tensors: {len(trainable)}")
print(f"Trainable tensors NOT under 'thinker.': "
      f"{[n for n,_ in trainable if not n.split('.',1)[-1].startswith('thinker') and 'thinker' not in n][:5]}")
assert not frozen_trainable_violation, f"VIOLATION: trainable params found in audio/vision/talker: {frozen_trainable_violation}"
print("\nPASS: zero trainable parameters under audio_tower/visual/talker/token2wav. "
      "All trainable parameters are LoRA adapters on the thinker's attention projections.")


trainable params: 7,372,800 || all params: 5,544,493,440 || trainable%: 0.1330

EXACT trainable parameter count: 7,372,800 / 5,544,493,440 total (0.1330%)
Number of distinct trainable parameter tensors: 288
Trainable tensors NOT under 'thinker.': []

PASS: zero trainable parameters under audio_tower/visual/talker/token2wav. All trainable parameters are LoRA adapters on the thinker's attention projections.


In [10]:
# ================= build one minimal training example (mirrors the repo's own train-mode ==
# masking scheme, main.py DynamicPromptCollator: labels=[-100]*len(prompt) + answer_ids) =====
QWEN_SYS = ("You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, "
            "capable of perceiving auditory and visual inputs, as well as generating text and speech.")

def build_training_inputs(rec, history_context):
    """Returns a dict of tensors (input_ids, attention_mask, labels, + audio features) for
    ONE example, ready for model(**inputs). Loss is masked to the completion (target word) only."""
    prompt_text = EVAL.build_prompt(rec["utterance"], history_context)
    wav_path = os.path.join(AUDIO_DIR, rec["id"] + ".wav")
    conv = [
        {"role": "system", "content": [{"type": "text", "text": QWEN_SYS}]},
        {"role": "user", "content": [
            {"type": "audio", "audio": wav_path},
            {"type": "text", "text": prompt_text}]},
    ]
    from qwen_omni_utils import process_mm_info
    rendered = proc.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)
    try:
        audios, images, videos = process_mm_info(conv, use_audio_in_video=False)
    except TypeError:
        audios, images, videos = process_mm_info(conv)
    try:
        prompt_inputs = proc(text=rendered, audio=audios, images=images, videos=videos,
                             return_tensors="pt", padding=False)
    except TypeError:
        prompt_inputs = proc(text=rendered, audios=audios, images=images, videos=videos,
                             return_tensors="pt", padding=False)

    prompt_ids = prompt_inputs["input_ids"][0]
    answer_ids = proc.tokenizer.encode(" " + rec["output"], add_special_tokens=False)
    eos_id = proc.tokenizer.eos_token_id
    answer_ids = torch.tensor(answer_ids + [eos_id], dtype=prompt_ids.dtype)

    full_ids = torch.cat([prompt_ids, answer_ids])
    labels = torch.cat([torch.full_like(prompt_ids, -100), answer_ids])
    attn = torch.ones_like(full_ids)

    out = dict(prompt_inputs)
    out["input_ids"] = full_ids.unsqueeze(0)
    out["attention_mask"] = attn.unsqueeze(0)
    out["labels"] = labels.unsqueeze(0)
    return out

# sanity print for one example (no model call yet)
_ex = build_training_inputs(nohist_tiny[0], None)
print("built one training example. input_ids shape:", _ex["input_ids"].shape,
      "labels shape:", _ex["labels"].shape,
      "n_masked(-100):", int((_ex["labels"][0] == -100).sum()),
      "n_supervised:", int((_ex["labels"][0] != -100).sum()))


built one training example. input_ids shape: torch.Size([1, 180]) labels shape: torch.Size([1, 180]) n_masked(-100): 178 n_supervised: 2


In [11]:
# ================= GATE 1: multi-T4 placement =================
print("="*78); print("GATE 1: multi-T4 placement"); print("="*78)
devices_used_now = sorted(set(str(p.device) for p in model.parameters()))
print("distinct devices holding model parameters:", devices_used_now)
GATE1_PASS = len([d for d in devices_used_now if d.startswith("cuda")]) >= 2
print(f"GATE1_PASS (>=2 distinct CUDA devices) = {GATE1_PASS}")
if not GATE1_PASS:
    print("NOTE: model landed on a single device. This alone is not a failure of training "
          "mechanics -- proceeding to forward/backward; multi-GPU-specific gates below will "
          "reflect single-device behavior if this is the case.")


GATE 1: multi-T4 placement
distinct devices holding model parameters: ['cuda:0', 'cuda:1']
GATE1_PASS (>=2 distinct CUDA devices) = True


In [12]:
# ================= GATE 2-3: forward loss + backward =================
print("="*78); print("GATE 2-3: forward loss + backward pass"); print("="*78)
model.train()

def to_model_device(inputs, ref_model=None):
    # NOTE: build_training_inputs() converts the processor's BatchFeature to a plain dict
    # (to splice in labels), which loses BatchFeature.to()'s built-in "cast only floating
    # tensors to model dtype, leave integer tensors alone" behavior that infer_one relies
    # on elsewhere in this project. Replicated manually here -- audio features (float) must
    # be cast to the model's dtype (fp16) or the audio encoder will see a dtype mismatch;
    # input_ids/attention_mask/labels (integer) must NOT be cast.
    m = ref_model if ref_model is not None else model
    first_param = next(m.parameters())
    device, dtype = first_param.device, first_param.dtype
    moved = {}
    for k, v in inputs.items():
        if torch.is_tensor(v):
            v = v.to(device)
            if torch.is_floating_point(v):
                v = v.to(dtype)
            moved[k] = v
        else:
            moved[k] = v
    return moved

GATE2_PASS = GATE3_PASS = False
ESCALATION_LOG = []
try:
    batch = to_model_device(_ex)
    outputs = model.thinker(**batch)
    loss = outputs.loss
    assert loss is not None and torch.isfinite(loss), f"loss invalid: {loss}"
    print(f"GATE2 forward/loss OK: loss={loss.item():.4f}")
    GATE2_PASS = True
    loss.backward()
    print("GATE3 backward OK: loss.backward() completed without error")
    GATE3_PASS = True
except Exception as e:
    ESCALATION_LOG.append(("device_map=auto", repr(e)))
    print(f"FAILURE at forward/backward with device_map='auto': {repr(e)}")
    print("Per the frozen escalation order, this would require rung 1 (hand-specified device map) "
          "or rung 2 (single-GPU) -- NOT implemented automatically in this cell; reported here for "
          "explicit human decision, not silently worked around.")
print(f"\nGATE2_PASS={GATE2_PASS}  GATE3_PASS={GATE3_PASS}")


GATE 2-3: forward loss + backward pass
GATE2 forward/loss OK: loss=3.9111
GATE3 backward OK: loss.backward() completed without error

GATE2_PASS=True  GATE3_PASS=True


## Gate-2 failure diagnostic (approved follow-up, NOT an escalation rung)

Localizes exactly where the forward call breaks: A) plain base-model control (same batch,
same devices, PEFT bypassed), B) inspection of the concrete classes/forward methods in the
PEFT call chain (does not infer from the exception text), C) direct forward-only calls
through the intermediate wrapper levels found in B, purely to locate the failing boundary.

Reuses the EXACT `_ex` batch and `to_model_device` helper already built for Gate 2 — no new
data, no config change. All calls below are forward-only (no `.backward()`, no optimizer
step) -- they cannot change any weight and are not a workaround; nothing here is proposed
as a fix. `target_modules`, `lora_config`, device_map, dtype, the processor, and the
training-example construction are all untouched.


In [13]:
# ================= DIAGNOSTIC A: plain base-model control =================
print("="*78); print("DIAGNOSTIC A: plain base_model forward (PEFT bypassed)"); print("="*78)
import inspect, traceback

print("base_model concrete class:", f"{type(base_model).__module__}.{type(base_model).__qualname__}")
print("base_model.forward defined by:",
      f"{base_model.forward.__module__}.{base_model.forward.__qualname__}")
try:
    print("base_model.forward signature:", inspect.signature(base_model.forward))
except Exception as e:
    print("(signature introspection failed:", e, ")")
print("\nNOTE: get_peft_model() already spliced LoRA adapters into base_model's Linear layers")
print("IN PLACE (same object graph 'model' also wraps) -- this call tests whether the ORIGINAL")
print("top-level forward() succeeds when invoked directly, bypassing PEFT's outer routing --")
print("not whether LoRA layers are present (they are, in both 'base_model' and 'model').")

A_PASS = False
A_loss = None
try:
    batch_a = to_model_device(_ex, ref_model=base_model)
    out_a = base_model(**batch_a)
    print("\nbase_model(**batch) SUCCEEDED.")
    print("  output type:", type(out_a))
    print("  has .loss attribute:", hasattr(out_a, "loss"))
    A_loss = getattr(out_a, "loss", None)
    print("  .loss value:", A_loss)
    A_PASS = torch.is_tensor(A_loss) and torch.isfinite(A_loss)
except Exception:
    print("\nbase_model(**batch) FAILED. Full traceback:")
    print(traceback.format_exc())
print(f"\nA_PASS (plain base_model returns a valid finite .loss) = {A_PASS}")


DIAGNOSTIC A: plain base_model forward (PEFT bypassed)
base_model concrete class: transformers.models.qwen2_5_omni.modeling_qwen2_5_omni.Qwen2_5OmniForConditionalGeneration
base_model.forward defined by: torch.nn.modules.module._forward_unimplemented
base_model.forward signature: (*input: Any) -> None

NOTE: get_peft_model() already spliced LoRA adapters into base_model's Linear layers
IN PLACE (same object graph 'model' also wraps) -- this call tests whether the ORIGINAL
top-level forward() succeeds when invoked directly, bypassing PEFT's outer routing --
not whether LoRA layers are present (they are, in both 'base_model' and 'model').

base_model(**batch) FAILED. Full traceback:
Traceback (most recent call last):
  File "/tmp/ipykernel_58/2028287841.py", line 21, in <cell line: 0>
    out_a = base_model(**batch_a)
            ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*

In [14]:
# ================= DIAGNOSTIC B: PEFT routing / class inspection =================
print("="*78); print("DIAGNOSTIC B: PEFT wrapper chain -- concrete classes and forward methods"); print("="*78)
import torch.nn as nn
from torch.nn.modules.module import _forward_unimplemented as _PLACEHOLDER_FWD

def describe(obj, label):
    cls = type(obj)
    fwd = getattr(obj, "forward", None)
    fwd_func = getattr(fwd, "__func__", fwd)
    is_placeholder = (fwd_func is _PLACEHOLDER_FWD)
    print(f"\n[{label}]")
    print(f"  concrete class      : {cls.__module__}.{cls.__qualname__}")
    print(f"  forward defined by  : {getattr(fwd_func, '__module__', '?')}.{getattr(fwd_func, '__qualname__', '?')}")
    print(f"  IS the raw placeholder forward (_forward_unimplemented) : {is_placeholder}")
    try:
        print(f"  forward signature   : {inspect.signature(fwd)}")
    except Exception as e:
        print(f"  (signature introspection failed: {e})")
    return is_placeholder

placeholder_model = describe(model, "model  (outer PEFT-wrapped object, what Gate 2 called)")

has_pb = hasattr(model, "base_model")
print(f"\nmodel has a '.base_model' attribute: {has_pb}")
placeholder_pb = False
if has_pb:
    placeholder_pb = describe(model.base_model, "model.base_model  (PEFT's internal tuner object, e.g. LoraModel)")

inner = None
placeholder_inner = False
if has_pb and hasattr(model.base_model, "model"):
    inner = model.base_model.model
    placeholder_inner = describe(inner, "model.base_model.model  (tuner's wrapped-model reference)")
    print(f"  is the SAME object as our 'base_model' notebook variable: {inner is base_model}")
else:
    print("\nmodel.base_model has no '.model' attribute")

if hasattr(model, "get_base_model"):
    gb = model.get_base_model()
    print(f"\nmodel.get_base_model() -> {type(gb).__module__}.{type(gb).__qualname__}  "
          f"| same object as base_model var: {gb is base_model}")

print("\n" + "-"*78)
print("WHICH OBJECT IN THE CHAIN IS THE RAW PLACEHOLDER (_forward_unimplemented)?")
print("-"*78)
print(f"  model                    : {placeholder_model}")
print(f"  model.base_model         : {placeholder_pb}")
print(f"  model.base_model.model   : {placeholder_inner}")


DIAGNOSTIC B: PEFT wrapper chain -- concrete classes and forward methods

[model  (outer PEFT-wrapped object, what Gate 2 called)]
  concrete class      : peft.peft_model.PeftModelForCausalLM
  forward defined by  : peft.peft_model.PeftModelForCausalLM.forward
  IS the raw placeholder forward (_forward_unimplemented) : False
  forward signature   : (input_ids=None, attention_mask=None, inputs_embeds=None, labels=None, output_attentions=None, output_hidden_states=None, return_dict=None, task_ids=None, **kwargs)

model has a '.base_model' attribute: True

[model.base_model  (PEFT's internal tuner object, e.g. LoraModel)]
  concrete class      : peft.tuners.lora.model.LoraModel
  forward defined by  : peft.tuners.tuners_utils.BaseTuner.forward
  IS the raw placeholder forward (_forward_unimplemented) : False
  forward signature   : (*args: 'Any', **kwargs: 'Any')

[model.base_model.model  (tuner's wrapped-model reference)]
  concrete class      : transformers.models.qwen2_5_omni.modeling_

In [15]:
# ================= DIAGNOSTIC C: forward-only calls at each wrapper boundary =================
# Localization only -- NOT a proposed fix. Forward-only (no .backward(), no optimizer.step()):
# cannot change any weight, cannot merge/unmerge adapters, cannot alter config.
print("="*78); print("DIAGNOSTIC C: same batch, called directly at each intermediate boundary"); print("="*78)

def try_call(obj, label, batch):
    print(f"\n--- {label} ---")
    try:
        out = obj(**batch)
        loss = getattr(out, "loss", None)
        ok = torch.is_tensor(loss) and torch.isfinite(loss)
        print(f"  SUCCESS. type(out)={type(out)}  loss={loss}  valid_finite_loss={ok}")
        return True, (float(loss) if ok else None)
    except Exception:
        print(f"  FAILED:\n{traceback.format_exc()}")
        return False, None

batch_c = to_model_device(_ex, ref_model=base_model)  # identical construction to Gate 2's batch
results = {}
if has_pb:
    results["model.base_model"] = try_call(model.base_model, "model.base_model (tuner)", batch_c)
if inner is not None:
    results["model.base_model.model"] = try_call(inner, "model.base_model.model (wrapped ref)", batch_c)
if hasattr(model, "get_base_model"):
    results["model.get_base_model()"] = try_call(model.get_base_model(), "model.get_base_model()", batch_c)

print("\n" + "="*78)
print("DIAGNOSTIC A/B/C SUMMARY")
print("="*78)
print(f"A: plain base_model(**batch) succeeded with valid loss = {A_PASS}")
print(f"B: raw-placeholder object in chain -> model={placeholder_model}  "
      f"model.base_model={placeholder_pb}  model.base_model.model={placeholder_inner}")
print("C: forward-only results at each boundary:")
for k, (ok, loss) in results.items():
    print(f"   {k}: success={ok}  loss={loss}")
print("\nNo workaround applied. Gates 3-7 not run. Awaiting interpretation before any next step.")


DIAGNOSTIC C: same batch, called directly at each intermediate boundary

--- model.base_model (tuner) ---
  FAILED:
Traceback (most recent call last):
  File "/tmp/ipykernel_58/183685995.py", line 9, in try_call
    out = obj(**batch)
          ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1787, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py", line 359, in forward
    return self.model(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^

In [16]:
# ================= DIAGNOSTIC D: class-level forward definition (bypasses any instance/hook) =================
# Read-only introspection of the CLASS's own __dict__ / MRO -- does not call, mutate, or
# reconfigure the model, device map, task_type, or target modules. Distinguishes:
#   (a) hook-timing bug: the class defines a real forward, but accelerate cached the wrong
#       reference before it was bound
#   (b) architecture: Qwen2_5OmniForConditionalGeneration never overrides forward at all
#       (generate() may dispatch straight to submodules' own .generate())
# NOT run yet -- proposed pending approval.
print("="*78); print("DIAGNOSTIC D: class-level forward definition, independent of any instance/hook"); print("="*78)
import inspect
from torch.nn.modules.module import _forward_unimplemented as _PLACEHOLDER_FWD

top_cls = type(base_model)
print(f"top-level class: {top_cls.__module__}.{top_cls.__qualname__}")
print(f"\nMRO (method resolution order):")
for c in top_cls.__mro__:
    has_own_forward = "forward" in c.__dict__
    marker = "  <-- defines its own 'forward'" if has_own_forward else ""
    print(f"  {c.__module__}.{c.__qualname__}{marker}")

owner = next(c for c in top_cls.__mro__ if "forward" in c.__dict__)
print(f"\ntop-level class's forward is supplied by: {owner.__module__}.{owner.__qualname__}")
print(f"is that literally torch.nn.modules.module._forward_unimplemented (by identity)? "
      f"{owner.__dict__['forward'] is _PLACEHOLDER_FWD}")

try:
    src_file = inspect.getsourcefile(owner.__dict__["forward"])
    print(f"source file: {src_file}")
except Exception as e:
    print(f"(source file lookup failed: {e})")

# Context only -- NOT a proposed training path. Just checking whether the thinker
# submodule's own class defines a forward, independent of the top-level class above.
if hasattr(base_model, "thinker"):
    thinker_cls = type(base_model.thinker)
    thinker_owner = next((c for c in thinker_cls.__mro__ if "forward" in c.__dict__), None)
    print(f"\n[context only] base_model.thinker concrete class: "
          f"{thinker_cls.__module__}.{thinker_cls.__qualname__}")
    if thinker_owner is not None:
        print(f"[context only] base_model.thinker's forward supplied by: "
              f"{thinker_owner.__module__}.{thinker_owner.__qualname__}")
        print(f"[context only] is thinker's forward the placeholder? "
              f"{thinker_owner.__dict__['forward'] is _PLACEHOLDER_FWD}")


DIAGNOSTIC D: class-level forward definition, independent of any instance/hook
top-level class: transformers.models.qwen2_5_omni.modeling_qwen2_5_omni.Qwen2_5OmniForConditionalGeneration

MRO (method resolution order):
  transformers.models.qwen2_5_omni.modeling_qwen2_5_omni.Qwen2_5OmniForConditionalGeneration
  transformers.models.qwen2_5_omni.modeling_qwen2_5_omni.Qwen2_5OmniPreTrainedModel
  transformers.modeling_utils.PreTrainedModel
  torch.nn.modules.module.Module  <-- defines its own 'forward'
  transformers.modeling_utils.EmbeddingAccessMixin
  transformers.modeling_utils.ModuleUtilsMixin
  transformers.utils.hub.PushToHubMixin
  transformers.integrations.peft.PeftAdapterMixin
  transformers.distributed.mixin.DistributedMixin
  transformers.generation.utils.GenerationMixin
  transformers.generation.continuous_batching.continuous_api.ContinuousMixin
  builtins.object

top-level class's forward is supplied by: torch.nn.modules.module.Module
is that literally torch.nn.modules.modu

In [17]:
# ================= DIAGNOSTIC E: thinker-level forward -- correct trainable path? =================
# Forward-only. No backward(), no optimizer.step(), no Gates 3-7, no Session 5, no protocol/
# target-module/hyperparameter/device-map/quantization change. Determines whether model.thinker
# is the correct object to call forward() on for training, and whether audio genuinely
# participates in that forward (not transcript-only fine-tuning).
print("="*78); print("DIAGNOSTIC E: thinker-level forward"); print("="*78)
import inspect, traceback, torch

# ---- E1: exact signature of model.thinker.forward ----
print("\n--- E1: model.thinker access + forward signature ---")
thinker_via_model = model.thinker
print(f"model.thinker resolves to: {type(thinker_via_model).__module__}.{type(thinker_via_model).__qualname__}")
print(f"model.thinker is base_model.thinker (from Diagnostic D)? {thinker_via_model is base_model.thinker}")

thinker_fwd_sig = inspect.signature(thinker_via_model.forward)
print(f"\nmodel.thinker.forward signature:\n  {thinker_fwd_sig}")
thinker_params = list(thinker_fwd_sig.parameters.keys())

# ---- E2: map existing batch fields onto that signature (report only, no discarding) ----
print("\n--- E2: batch field <-> forward-parameter mapping (no fields dropped) ---")
batch_e = to_model_device(_ex, ref_model=thinker_via_model)
for k, v in batch_e.items():
    shape = tuple(v.shape) if torch.is_tensor(v) else None
    dtype = v.dtype if torch.is_tensor(v) else type(v)
    device = v.device if torch.is_tensor(v) else None
    in_sig = k in thinker_params
    print(f"  {k:28s} shape={str(shape):22s} dtype={str(dtype):18s} device={str(device):10s} "
          f"in forward signature: {in_sig}")

unmatched = [k for k in batch_e if k not in thinker_params]
print(f"\nbatch fields NOT present as named parameters of model.thinker.forward "
      f"(may still be absorbed via **kwargs -- checked next): {unmatched}")
accepts_var_kwargs = any(p.kind == inspect.Parameter.VAR_KEYWORD for p in thinker_fwd_sig.parameters.values())
print(f"model.thinker.forward accepts **kwargs: {accepts_var_kwargs}")

# ---- E3a: structural check -- is there an audio placeholder token in input_ids? ----
print("\n--- E3a: structural audio-token check (config-driven, no hardcoded token id) ---")
audio_token_id = None
for cfg_obj, cfg_label in [(getattr(thinker_via_model, "config", None), "thinker.config"),
                           (getattr(base_model, "config", None), "base_model.config")]:
    if cfg_obj is None:
        continue
    for attr_name in ["audio_token_id", "audio_token_index", "audio_start_token_id", "audio_token"]:
        val = getattr(cfg_obj, attr_name, None)
        if isinstance(val, int):
            audio_token_id = val
            print(f"  found {cfg_label}.{attr_name} = {val}")
            break
    if audio_token_id is not None:
        break
if audio_token_id is None:
    print("  could not locate an integer audio-token-id field on config under common attribute "
          "names -- skipping this specific structural check (not fabricating one).")
else:
    n_occurrences = int((batch_e["input_ids"][0] == audio_token_id).sum())
    print(f"  occurrences of audio_token_id={audio_token_id} in this example's input_ids: {n_occurrences}")

# ---- E3b: source-level check -- are the audio fields referenced in forward's body? ----
print("\n--- E3b: source-level reference check ---")
audio_field_candidates = [k for k, v in batch_e.items()
                          if k not in ("input_ids", "attention_mask", "labels")
                          and torch.is_tensor(v) and torch.is_floating_point(v)]
print(f"candidate audio/vision floating-point fields in batch: {audio_field_candidates}")
try:
    src = inspect.getsource(type(thinker_via_model).forward)
    for field in audio_field_candidates:
        hits = [i for i, line in enumerate(src.splitlines()) if field in line]
        preview = (" -- e.g.: " + src.splitlines()[hits[0]].strip()) if hits else ""
        print(f"  '{field}' referenced on {len(hits)} line(s) inside forward()'s source{preview}")
except Exception as e:
    print(f"  (source inspection unavailable: {e})")

# ---- E4: empirical sensitivity probe (forward-only, real audio vs. perturbed audio) ----
print("\n--- E4: empirical audio-sensitivity probe (forward-only, no backward) ---")
model.eval()  # deterministic; avoids LoRA-dropout noise confounding the comparison
with torch.no_grad():
    out_real = thinker_via_model(**batch_e)
    loss_real = out_real.loss
    print(f"forward with REAL audio features: loss = {loss_real.item():.6f}")

    batch_perturbed = dict(batch_e)
    g = torch.Generator(device="cpu").manual_seed(0)
    for field in audio_field_candidates:
        real_t = batch_e[field]
        noise = torch.randn(tuple(real_t.shape), generator=g, dtype=torch.float32).to(real_t.device, real_t.dtype)
        batch_perturbed[field] = noise
        print(f"  replaced '{field}' with random noise, same shape/dtype/device "
              f"(mean|real|={real_t.abs().mean().item():.4f}, mean|noise|={noise.abs().mean().item():.4f})")

    out_perturbed = thinker_via_model(**batch_perturbed)
    loss_perturbed = out_perturbed.loss
    print(f"forward with PERTURBED audio features: loss = {loss_perturbed.item():.6f}")

loss_delta = abs(loss_real.item() - loss_perturbed.item())
print(f"\n|loss_real - loss_perturbed| = {loss_delta:.6f}")
AUDIO_CONSUMED = loss_delta > 1e-4
print(f"AUDIO_CONSUMED (loss changes materially when only audio is perturbed) = {AUDIO_CONSUMED}")
model.train()

# ---- E5: the actual Gate-2-style forward+loss call this diagnostic is validating ----
print("\n--- E5: thinker-level forward + loss (the call this would replace Gate 2 with) ---")
E_PASS = False
try:
    batch_e2 = to_model_device(_ex, ref_model=thinker_via_model)
    out_e = thinker_via_model(**batch_e2)
    loss_e = out_e.loss
    print(f"model.thinker(**batch) call used, exact kwargs: {sorted(batch_e2.keys())}")
    print(f"output type: {type(out_e)}")
    print(f"loss: {loss_e}")
    E_PASS = torch.is_tensor(loss_e) and torch.isfinite(loss_e)
except Exception:
    print("model.thinker(**batch) FAILED. Full traceback:")
    print(traceback.format_exc())
print(f"\nE_PASS (thinker forward returns valid finite loss) = {E_PASS}")

# ---- E6: identity check -- are the LoRA-adapted layers the SAME objects via this path? ----
print("\n--- E6: LoRA parameter identity check (model.<name> vs model.thinker.<relative_name>) ---")
mismatches = []
not_lora = []
checked = 0
for name in TARGET_MODULES:
    assert name.startswith("thinker.")
    rel_name = name[len("thinker."):]
    try:
        mod_via_model = model.get_submodule(name)
        mod_via_thinker = thinker_via_model.get_submodule(rel_name)
    except AttributeError as e:
        mismatches.append((name, f"get_submodule failed: {e}"))
        continue
    checked += 1
    if mod_via_model is not mod_via_thinker:
        mismatches.append((name, "NOT the same object"))
    if not (hasattr(mod_via_model, "lora_A") and hasattr(mod_via_model, "lora_B")):
        not_lora.append(name)

print(f"checked {checked} / {len(TARGET_MODULES)} target modules")
print(f"identity mismatches (model.<name> is NOT model.thinker.<rel_name>): {len(mismatches)}")
if mismatches:
    for m in mismatches[:10]:
        print(f"    {m}")
print(f"modules missing lora_A/lora_B (not actually LoRA-adapted): {len(not_lora)}")
if not_lora:
    for n in not_lora[:10]:
        print(f"    {n}")
E6_IDENTITY_OK = (checked == len(TARGET_MODULES)) and not mismatches and not not_lora
print(f"\nE6_IDENTITY_OK (every target module reachable via model.thinker and genuinely LoRA-adapted) "
      f"= {E6_IDENTITY_OK}")

# ---- summary ----
print("\n" + "="*78)
print("DIAGNOSTIC E SUMMARY")
print("="*78)
print(f"E_PASS (thinker forward -> valid finite loss)              = {E_PASS}")
print(f"AUDIO_CONSUMED (loss sensitive to audio perturbation)       = {AUDIO_CONSUMED}")
print(f"E6_IDENTITY_OK (LoRA params identical via thinker path)     = {E6_IDENTITY_OK}")
print("\nNo backward pass run. No optimizer step. Gates 3-7 not run. Session 5 not accessed.")
print("No protocol, target-module, hyperparameter, device-map, or quantization change made.")


DIAGNOSTIC E: thinker-level forward

--- E1: model.thinker access + forward signature ---
model.thinker resolves to: transformers.models.qwen2_5_omni.modeling_qwen2_5_omni.Qwen2_5OmniThinkerForConditionalGeneration
model.thinker is base_model.thinker (from Diagnostic D)? True

model.thinker.forward signature:
  (input_ids: torch.LongTensor | None = None, input_features: torch.FloatTensor | None = None, pixel_values: torch.FloatTensor | None = None, pixel_values_videos: torch.FloatTensor | None = None, image_grid_thw: torch.LongTensor | None = None, video_grid_thw: torch.LongTensor | None = None, attention_mask: torch.Tensor | None = None, feature_attention_mask: torch.Tensor | None = None, audio_feature_lengths: torch.LongTensor | None = None, position_ids: torch.LongTensor | None = None, past_key_values: transformers.cache_utils.Cache | None = None, inputs_embeds: torch.FloatTensor | None = None, labels: torch.LongTensor | None = None, use_cache: bool | None = None, use_audio_in_video

In [18]:
# ================= GATE 4: LoRA-only gradients =================
print("="*78); print("GATE 4: gradients exist ONLY on LoRA parameters"); print("="*78)
GATE4_PASS = None
if GATE3_PASS:
    bad_frozen_with_grad = []
    lora_params_with_grad = []
    lora_params_without_grad = []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            if p.grad is not None and p.grad.abs().sum().item() != 0:
                bad_frozen_with_grad.append(n)
        else:
            if p.grad is None or p.grad.abs().sum().item() == 0:
                lora_params_without_grad.append(n)
            else:
                lora_params_with_grad.append(n)
    print(f"trainable params WITH nonzero grad: {len(lora_params_with_grad)}")
    print(f"trainable params WITHOUT grad (unexpected if >0): {len(lora_params_without_grad)}")
    print(f"frozen params that unexpectedly received a grad (must be 0): {len(bad_frozen_with_grad)}")
    GATE4_PASS = (len(bad_frozen_with_grad) == 0) and (len(lora_params_with_grad) > 0)
    print(f"GATE4_PASS = {GATE4_PASS}")
else:
    print("SKIPPED (GATE3 did not pass)")


GATE 4: gradients exist ONLY on LoRA parameters
trainable params WITH nonzero grad: 144
trainable params WITHOUT grad (unexpected if >0): 144
frozen params that unexpectedly received a grad (must be 0): 0
GATE4_PASS = True


In [19]:
# ================= GATE 5: optimizer step actually updates LoRA weights =================
print("="*78); print("GATE 5: optimizer step updates LoRA weights"); print("="*78)
GATE5_PASS = None
optimizer = None
if GATE4_PASS:
    trainable_params = [p for _, p in model.named_parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=1e-4, betas=(0.9, 0.95))
    probe_name, probe_param = next((n, p) for n, p in model.named_parameters() if p.requires_grad)
    before = probe_param.detach().clone()
    optimizer.step()
    optimizer.zero_grad()
    after = probe_param.detach().clone()
    changed = not torch.equal(before, after)
    print(f"probe param: {probe_name}  changed_after_step={changed}  "
          f"max_abs_delta={float((after-before).abs().max()):.6g}")
    GATE5_PASS = changed
    print(f"GATE5_PASS = {GATE5_PASS}")
else:
    print("SKIPPED (GATE4 did not pass)")


GATE 5: optimizer step updates LoRA weights
probe param: base_model.model.thinker.model.layers.0.self_attn.q_proj.lora_A.default.weight  changed_after_step=True  max_abs_delta=2.23517e-08
GATE5_PASS = True


In [20]:
# ================= GATE 6: checkpoint save -> fresh load -> resume =================
print("="*78); print("GATE 6: checkpoint save / fresh load / resume"); print("="*78)
GATE6_PASS = None
ADAPTER_DIR = os.path.join(OUTPUT_DIR, "adapter_smoketest")
OPT_PATH = os.path.join(OUTPUT_DIR, "optimizer_smoketest.pt")
if GATE5_PASS:
    try:
        # save adapter weights + optimizer state
        model.save_pretrained(ADAPTER_DIR)
        torch.save(optimizer.state_dict(), OPT_PATH)
        saved_lora_snapshot = {n: p.detach().clone().cpu()
                               for n, p in model.named_parameters() if p.requires_grad}
        print(f"saved adapter to {ADAPTER_DIR} and optimizer state to {OPT_PATH}")

        # fresh load: new PEFT wrapper instance around the SAME base model, loading the saved adapter
        from peft import PeftModel
        # drop the current LoRA wrapper's weights conceptually by loading into a NEW peft model object
        fresh_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR, is_trainable=True)
        reload_ok = True
        for n, p in fresh_model.named_parameters():
            if p.requires_grad:
                if n not in saved_lora_snapshot:
                    reload_ok = False; print("MISSING in snapshot:", n); continue
                if not torch.equal(p.detach().cpu(), saved_lora_snapshot[n]):
                    reload_ok = False; print("MISMATCH after reload:", n)
        print(f"reloaded adapter weights match saved snapshot exactly: {reload_ok}")

        # resume: new optimizer, restore state, take one more step
        fresh_trainable = [p for _, p in fresh_model.named_parameters() if p.requires_grad]
        fresh_optimizer = torch.optim.AdamW(fresh_trainable, lr=1e-4, betas=(0.9, 0.95))
        fresh_optimizer.load_state_dict(torch.load(OPT_PATH, map_location="cpu"))
        fresh_model.train()
        batch2 = to_model_device(build_training_inputs(nohist_tiny[1 % len(nohist_tiny)], None),
                                 ref_model=fresh_model)
        out2 = fresh_model.thinker(**batch2)
        out2.loss.backward()
        fresh_optimizer.step()
        fresh_optimizer.zero_grad()
        print("resumed training step after reload completed without error")

        GATE6_PASS = reload_ok
        model = fresh_model  # continue using the reloaded model for Gate 7
    except Exception as e:
        ESCALATION_LOG.append(("checkpoint/resume", repr(e)))
        print(f"FAILURE in checkpoint/resume: {repr(e)}")
        GATE6_PASS = False
    print(f"\nGATE6_PASS = {GATE6_PASS}")
else:
    print("SKIPPED (GATE5 did not pass)")


GATE 6: checkpoint save / fresh load / resume
saved adapter to /kaggle/working/phase3_smoketest/adapter_smoketest and optimizer state to /kaggle/working/phase3_smoketest/optimizer_smoketest.pt


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


reloaded adapter weights match saved snapshot exactly: True
resumed training step after reload completed without error

GATE6_PASS = True


In [21]:
# ================= GATE 7: post-training generation still works =================
print("="*78); print("GATE 7: post-training generate() still functions"); print("="*78)
GATE7_PASS = None
if GATE6_PASS is True:  # strict: only proceed if the full chain up to GATE6 actually PASSED,
                        # not merely "wasn't None" -- a skipped (None) upstream gate must not
                        # silently let a downstream gate run and be reported as if independent.
    try:
        model.eval()
        rec = nohist_tiny[0]
        prompt = EVAL.build_prompt(rec["utterance"], None)
        wav_path = os.path.join(AUDIO_DIR, rec["id"] + ".wav")
        conv = [
            {"role": "system", "content": [{"type": "text", "text": QWEN_SYS}]},
            {"role": "user", "content": [
                {"type": "audio", "audio": wav_path},
                {"type": "text", "text": prompt}]},
        ]
        from qwen_omni_utils import process_mm_info
        text = proc.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)
        audios, images, videos = process_mm_info(conv, use_audio_in_video=False)
        inputs = proc(text=text, audio=audios, images=images, videos=videos,
                      return_tensors="pt", padding=True)
        inputs = to_model_device(inputs)
        with torch.no_grad():
            try:
                out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                     return_audio=False)
            except TypeError:
                out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        if isinstance(out, (tuple, list)):
            out = out[0]
        gen = out[:, inputs["input_ids"].shape[1]:]
        raw = proc.batch_decode(gen, skip_special_tokens=True)[0].strip()
        print(f"post-training generation OK -> id={rec['id']} gold={rec['output']} raw={raw!r}")
        GATE7_PASS = len(raw.strip()) > 0
    except Exception as e:
        ESCALATION_LOG.append(("post-training generate", repr(e)))
        print(f"FAILURE in post-training generate(): {repr(e)}")
        GATE7_PASS = False
    print(f"\nGATE7_PASS = {GATE7_PASS}")
else:
    print("SKIPPED (GATE6 did not pass)")


GATE 7: post-training generate() still functions
post-training generation OK -> id=Ses01F_impro03_F012 gold=happy raw='Neutral. What do you think about this? Do you have any other utterances to analyze?'

GATE7_PASS = True


In [22]:
# ================= SUMMARY =================
print("="*78)
print("PHASE-3 ARCHITECTURE INSPECTION + SMOKE TEST -- SUMMARY")
print("="*78)
gates = {
    "GATE1 multi-T4 placement": GATE1_PASS,
    "GATE2 forward/loss": GATE2_PASS,
    "GATE3 backward pass": GATE3_PASS,
    "GATE4 LoRA-only gradients": GATE4_PASS,
    "GATE5 optimizer step updates weights": GATE5_PASS,
    "GATE6 checkpoint save/fresh-load/resume": GATE6_PASS,
    "GATE7 post-training generation": GATE7_PASS,
}
for name, val in gates.items():
    print(f"  {name:<42} {val}")

print(f"\nTARGET_MODULES: {len(TARGET_MODULES)} fully-qualified thinker attention paths")
print(f"exact trainable parameters: {n_trainable:,} / {n_total:,} ({100*n_trainable/n_total:.4f}%)")
print(f"zero trainable params under audio/vision/talker: "
      f"{'CONFIRMED' if not frozen_trainable_violation else 'VIOLATED'}")

if ESCALATION_LOG:
    print("\nESCALATION LOG (failures encountered, not silently worked around):")
    for stage, err in ESCALATION_LOG:
        print(f"  [{stage}] {err}")

all_required = [GATE1_PASS, GATE2_PASS, GATE3_PASS, GATE4_PASS, GATE5_PASS, GATE6_PASS, GATE7_PASS]
cleared = all(v is True for v in all_required)
print("\n" + "="*78)
print(f"CLEARED FOR FULL TRAINING: {'YES' if cleared else 'NO'}")
print("="*78)
if not cleared:
    print("At least one gate did not pass. Per the frozen protocol, do not proceed to full")
    print("S1-3 training. Report exactly which gate failed and the escalation log above before")
    print("any next step is decided.")

import json as _json
summary = {"target_modules_count": len(TARGET_MODULES), "target_modules": TARGET_MODULES,
           "trainable_params": n_trainable, "total_params": n_total,
           "devices_used": devices_used_now, "gates": gates,
           "escalation_log": ESCALATION_LOG, "cleared_for_full_training": cleared}
with open(os.path.join(OUTPUT_DIR, "phase3_smoketest_summary.json"), "w", encoding="utf-8") as f:
    _json.dump(summary, f, ensure_ascii=False, indent=2, default=str)
print(f"\nwrote {OUTPUT_DIR}/phase3_smoketest_summary.json")
print("\nSave Version -> Save & Run All (Commit) before closing, so this summary and the full")
print("printed log survive for review.")


PHASE-3 ARCHITECTURE INSPECTION + SMOKE TEST -- SUMMARY
  GATE1 multi-T4 placement                   True
  GATE2 forward/loss                         True
  GATE3 backward pass                        True
  GATE4 LoRA-only gradients                  True
  GATE5 optimizer step updates weights       True
  GATE6 checkpoint save/fresh-load/resume    True
  GATE7 post-training generation             True

TARGET_MODULES: 144 fully-qualified thinker attention paths
exact trainable parameters: 7,372,800 / 5,544,493,440 (0.1330%)
zero trainable params under audio/vision/talker: CONFIRMED

CLEARED FOR FULL TRAINING: YES

wrote /kaggle/working/phase3_smoketest/phase3_smoketest_summary.json

Save Version -> Save & Run All (Commit) before closing, so this summary and the full
printed log survive for review.


## Not done here (by design)

No full Session-1-3 training job was started. No Session-5 file was read, written, or
referenced anywhere above. If `CLEARED FOR FULL TRAINING` printed `NO`, the next step is a
human decision (per the frozen escalation order) — not an automatic retry with a different
architecture, quantization, or target-module set.
